# 📊 Notebook 04 — Ewaluacja i Walidacja
## Model Evaluation and Validation

This notebook provides a comprehensive evaluation of both trained models:
- Confusion matrices (normalized)
- ROC / AUC curves
- **False negative analysis** — clinically critical (missed falls = patient risk)
- Sensitivity / Specificity trade-off
- Temporal prediction timeline on a test sequence

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
plt.style.use('dark_background')
Path('../reports').mkdir(exist_ok=True)
SEED = 42
WINDOW_SIZE = 30
STEP = 15
print('Setup complete.')

## 1. Load Test Data

In [ ]:
from src.data.generator import SyntheticGenerator
from src.data.features import FeatureExtractor

# Different seed from training — truly held-out test set
df_test = SyntheticGenerator(seed=SEED + 99).generate(n_normal=200, n_anomaly=100)
print(f'Test dataset: {len(df_test):,} rows')

fe = FeatureExtractor()
X_frame, y_frame = fe.frame_features(df_test, label_col='label')
X_seq, y_seq = fe.window_features(df_test, label_col='label', window_size=WINDOW_SIZE, step=STEP)

_, X_te, _, y_te = train_test_split(X_frame.values, y_frame.values, test_size=0.5, stratify=y_frame.values, random_state=SEED)
_, X_seq_te, _, y_seq_te = train_test_split(X_seq, y_seq, test_size=0.5, stratify=y_seq, random_state=SEED)

print(f'Frame test set: {len(y_te)} samples (Anomaly: {y_te.sum()})')
print(f'Window test set: {len(y_seq_te)} samples (Anomaly: {y_seq_te.sum()})')

## 2. Load Trained Models

In [ ]:
from src.models.random_forest import FallRandomForest
from src.models.lstm import FallLSTM

rf_path   = Path('../models/random_forest.pkl')
lstm_path = Path('../models/lstm_model.keras')

y_pred_rf = y_proba_rf = y_pred_lstm = y_proba_lstm = None

if rf_path.exists():
    rf = FallRandomForest.load(str(rf_path))
    y_pred_rf  = rf.predict(X_te)
    y_proba_rf = rf.predict_proba(X_te)
    print(f'Random Forest loaded. Test accuracy: {(y_pred_rf == y_te).mean():.3f}')
else:
    print('RF model not found — run Notebook 03 first.')

if lstm_path.exists():
    lstm = FallLSTM.load(str(lstm_path))
    y_pred_lstm  = lstm.predict(X_seq_te)
    y_proba_lstm = lstm.predict_proba(X_seq_te)
    print(f'LSTM loaded. Test accuracy: {(y_pred_lstm == y_seq_te).mean():.3f}')
else:
    print('LSTM model not found — run Notebook 03 first.')

## 3. Confusion Matrices

In [ ]:
from src.evaluation.metrics import Evaluator
ev = Evaluator(save_dir='../reports', show_plots=True)

if y_pred_rf is not None:
    ev.confusion_matrix_plot(y_te, y_pred_rf, title='Random Forest')
if y_pred_lstm is not None:
    ev.confusion_matrix_plot(y_seq_te, y_pred_lstm, title='LSTM')

## 4. ROC Curves

In [ ]:
if y_proba_rf is not None or y_proba_lstm is not None:
    y_true_roc = y_te
    proba_rf_roc = y_proba_rf
    proba_lstm_roc = None
    if y_proba_lstm is not None:
        min_len = min(len(y_te), len(y_seq_te))
        y_true_roc = y_te[:min_len]
        proba_rf_roc = y_proba_rf[:min_len] if y_proba_rf is not None else None
        proba_lstm_roc = y_proba_lstm[:min_len]
    ev.roc_plot(y_true_roc, proba_rf_roc, proba_lstm_roc)

## 5. ⚠ False Negative Analysis (Missed Falls — Clinical Priority)

In [ ]:
if y_pred_rf is not None:
    ev.false_negative_analysis(y_te, y_pred_rf, 'Random Forest')
if y_pred_lstm is not None:
    ev.false_negative_analysis(y_seq_te, y_pred_lstm, 'LSTM')

## 6. Metrics Summary Table

In [ ]:
if y_pred_rf is not None:
    min_len = min(len(y_te), len(y_seq_te)) if y_pred_lstm is not None else len(y_te)
    metrics = ev.summary_table(
        y_true=y_te[:min_len],
        y_pred_rf=y_pred_rf[:min_len],
        y_pred_lstm=y_pred_lstm[:min_len] if y_pred_lstm is not None else None,
        y_proba_rf=y_proba_rf[:min_len] if y_proba_rf is not None else None,
        y_proba_lstm=y_proba_lstm[:min_len] if y_proba_lstm is not None else None,
    )
    metrics.to_csv('../reports/metrics_summary.csv')
    print('Saved: reports/metrics_summary.csv')
    metrics

## 7. Temporal Prediction Timeline

In [ ]:
# Visualise model predictions over time for a single test sequence
if y_pred_rf is not None:
    n_frames = min(300, len(y_te))
    t = np.arange(n_frames)

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    fig.suptitle('Temporal Prediction Timeline', color='white', fontsize=13)
    fig.patch.set_facecolor('#0d0d1a')

    for ax, true_lbl, pred_lbl, proba, title in [
        (axes[0], y_te[:n_frames], y_pred_rf[:n_frames], y_proba_rf[:n_frames, 1], 'Random Forest'),
    ]:
        ax.fill_between(t, true_lbl, alpha=0.3, color='#4CAF50', label='Ground Truth')
        ax.plot(t, proba, color='#F44336', lw=1.5, label='Anomaly Probability')
        ax.axhline(0.5, color='#FF9800', linestyle='--', lw=1, label='Decision Threshold')
        ax.set_title(title, color='white')
        ax.set_ylabel('Score', color='#aaa')
        ax.set_ylim(-0.05, 1.15)
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='#aaa')
        ax.legend(fontsize=8)

    # Mark false negatives in red
    fn_mask = (y_te[:n_frames] == 1) & (y_pred_rf[:n_frames] == 0)
    if fn_mask.any():
        axes[0].scatter(t[fn_mask], np.ones(fn_mask.sum()) * 0.5,
                        color='#FF1744', s=40, zorder=5, label='Missed Fall (FN)')
        axes[0].legend(fontsize=8)

    axes[-1].set_xlabel('Frame', color='#aaa')
    if len(axes) > 1:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.savefig('../reports/temporal_predictions.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
    plt.show()
    print('Saved: reports/temporal_predictions.png')